# Model 04: Inventory Status Prediction
### College Data Mining & DWDM Laboratory Submission
**Algorithm:** Logistic Regression  
**Problem Type:** Multiclass Classification with Feature Scaling  
**Key Concepts Demonstrated:**
- Syllabus-aligned algorithm implementation
- Data preprocessing & descriptive statistics
- Model training, hyperparameter tuning & evaluation
- Joblib model serialization and sample inference


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Ensure parent directory is in path
sys.path.append(os.path.abspath(".."))
from preprocessing.preprocess import load_dataset, clean_and_impute

# Load and clean dataset
df = load_dataset()
df_clean = clean_and_impute(df)
print(f"Dataset ready: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head(3)


Dataset ready: 1000 rows, 30 columns


,date,day_of_week,is_weekend,product_name,category,price,discount,previous_day_sales,sales_7_day_average,sales_30_day_average,...,sales_90_day,turnover_rate,days_since_last_sale,category_label,inventory_status,business_activity,total_expense,expense_3_month_average,average_order_value,profit
0,2024-01-01,Monday,0,Laundry Detergent 1kg,Household,210.0,10.43,3599.34,3599.34,3599.34,...,205357.08,3.62,12,MEDIUM_DEMAND,RESTOCK,14.87,3044.67,3044.67,97.28,-275.43
1,2024-01-02,Tuesday,0,Basmati Rice 5kg,Grocery,450.0,2.96,3599.34,3656.81,3656.81,...,316258.19,3.77,1,MEDIUM_DEMAND,RESTOCK,15.39,3100.65,3072.66,95.24,-358.75
2,2024-01-03,Wednesday,0,Organic Honey 500g,Grocery,280.0,6.88,3714.28,3750.13,3750.13,...,159407.77,5.20,0,MEDIUM_DEMAND,RESTOCK,12.33,3445.75,3197.02,115.79,-448.90


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

feature_cols = ["current_stock", "predicted_demand", "reorder_level", "quantity_sold", "turnover_rate"]
X = df_clean[feature_cols].values
y = df_clean["inventory_status"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=500, random_state=42))
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Accuracy:       {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")


Accuracy:       97.00%
Macro F1-Score: 0.9764


In [3]:
# Confusion Matrix
labels = ["RESTOCK", "SUFFICIENT", "OVERSTOCK"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Actual {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
cm_df


,Pred RESTOCK,Pred SUFFICIENT,Pred OVERSTOCK
Actual RESTOCK,96,1,0
Actual SUFFICIENT,5,71,0
Actual OVERSTOCK,0,0,27


In [4]:
# Joblib Save & Test
joblib.dump(pipeline, "../models/inventory_model.joblib")
del pipeline
loaded_pipe = joblib.load("../models/inventory_model.joblib")

sample = np.array([[15, 100, 40, 90, 2.2]])
status = loaded_pipe.predict(sample)[0]
print(f"Predicted Inventory Status: {status}")


Predicted Inventory Status: RESTOCK
